<font color="#D35400" size='6px'><b>Image Semantic Edge (Boundary Analysis)</b></font>

<font color="#D35400" size='4px'><b>The Pipeline Architecture</b></font>

The system operates in a morphological pipeline designed to extract artistic or analytical "sketch" views of the image by isolating the precise boundaries of detected objects. It moves beyond simple edge detection (like Canny) by leveraging **SAM 3** to understand object semantics before drawing edges .

<font color="#D35400" size='4px'><b>Stage I: Scene Understanding & Class Discovery</b></font>
The system first identifies what objects are present to ensure edges are only drawn for meaningful entities, not noise.

**Automatic Discovery:** The image is analyzed by a Vision-Language Model.
* <font color="#884EA0"><i>LLM Mode:</i></font> Uses **Async Parallel Processing** to analyze multiple crops, detecting complex structures (e.g., "gothic architecture details").
* <font color="#884EA0"><i>BLIP Mode:</i></font> Uses local captioning to find standard objects (e.g., "building", "window").

**NLP Refinement:** The detected nouns are cleaned and deduped via **SpaCy** to form a precise target list.

<font color="#D35400" size='4px'><b>Stage II: SAM 3 Instance Segmentation</b></font>
Edges are derived from masks, not raw pixels.

**Mask Generation:** SAM 3 predicts high-quality binary **Spatial Masks** for every target object.

**Semantic Separation:** Unlike standard edge filters that pick up texture noise (like grass blades), this stage ensures edges are only generated for the *boundaries* of recognized objects (e.g., the outline of a cat, but not its fur pattern).

<font color="#D35400" size='4px'><b>Stage III: Morphological Edge Extraction</b></font>
The solid masks are mathematically converted into outlines.

**Morphological Gradient:** The system applies `cv2.morphologyEx` with a **Gradient** kernel.
* <font color="#884EA0"><i>Dilation:</i></font> Expands the mask slightly.
* <font color="#884EA0"><i>Erosion:</i></font> Shrinks the mask slightly.
* <font color="#884EA0"><i>Subtraction:</i></font> The difference between the dilated and eroded masks leaves a thin, perfect boundary line.

**Kernel Control:** The `thickness` parameter controls the size of the kernel, allowing for fine lines (pencil sketch style) or thick borders (marker style).

<font color="#D35400" size='4px'><b>Stage IV: Visualization & Rendering</b></font>
The final output is rendered in two distinct modes.

**Sketch Mode (B&W):** A high-contrast binary map showing white edges on a black background, ideal for structural analysis.

**Colored Mode:** Edges are colored according to their object class (e.g., "Person" = Red outline, "Car" = Blue outline), creating a neon-like semantic visualization.

<font color="#2E86C1" size='6px'>Environment Initialization & Repository Setup</font>

In [ ]:
COLAB = False

In [ ]:
import os
import sys

if COLAB:
    # Clone Repository 
    if not os.path.exists("sam3-toolkit"):
        print("--- Cloning repository...")
        !git clone https://github.com/MrKGhasemi/sam3-toolkit.git
    else:
        print("--- Repository already exists.")

    # Auto-Detect Paths 
    repo_root = os.path.abspath("sam3-toolkit")
    sam3_install_path = None
    practical_path = None

    for root, dirs, files in os.walk(repo_root):
        if ("sam3" in dirs or "sam3_main" in root or root == repo_root):
            if sam3_install_path is None or len(root) < len(sam3_install_path):
                sam3_install_path = root

        if "semantic_edge.py" in files:
            practical_path = root

    if not sam3_install_path or not practical_path:
        raise FileNotFoundError(
            "     --- Critical paths (sam3 or practical) not found.")

    print(f"Found SAM 3 Library at: {sam3_install_path}")
    print(f"Found Project Code at: {practical_path}")

    # Install Dependencies 
    print("--- Installing Python packages...")
    !{sys.executable} -m pip install -q opencv-python matplotlib scikit-learn transformers spacy openai gdown mega.py huggingface_hub
    !{sys.executable} -m pip install -q einops decord pycocotools

    # Install SAM 3 
    print("--- Installing SAM 3 Library...")
    !{sys.executable} -m pip install -e "{sam3_install_path}"

    # Configure Working Directory 
    os.chdir(practical_path)
    if practical_path not in sys.path:
        sys.path.insert(0, practical_path)
    print(f"--- Working directory set to: {os.getcwd()}")

    # Download Model Weights
    weights_dir = "weights"
    weights_path = os.path.join(weights_dir, "sam3.pt")
    os.makedirs(weights_dir, exist_ok=True)

    # If the model is Private/Gated, you MUST provide a token.
    # Get it here: https://huggingface.co/settings/tokens
    HF_REPO_ID = "facebook/sam3" 
    HF_FILENAME = "sam3.pt"               
    HF_TOKEN = "HF_TOKEN"  # Replace with your actual token or set to None

    if not os.path.exists(weights_path):
        print(f"--- Starting download...")

        try:
            from huggingface_hub import hf_hub_download
            print("--- Connecting to Hugging Face Hub...")
            downloaded_file = hf_hub_download(
                repo_id=HF_REPO_ID,
                filename=HF_FILENAME,
                local_dir=weights_dir,
                token=HF_TOKEN,
                local_dir_use_symlinks=False  # Ensure we get the actual file, not a symlink
            )
            if os.path.basename(downloaded_file) != "sam3.pt":
                os.rename(downloaded_file, weights_path)

            print("--- Download attempt finished.")

        except Exception as e:
            print(f"      --- Error downloading: {e}")
            print("           Tip: If using Hugging Face private repo, ensure HF_TOKEN is set.")

    else:
        print("--- Weights file already exists.")

    # Verify File Integrity 
    if os.path.exists(weights_path):
        final_size = os.path.getsize(weights_path) / (1024 * 1024)
        print(f"--- Final File Size: {final_size:.2f} MB")
        if final_size < 2000:
            print(
                "      --- WARNING: File seems too small (<2GB). It might be corrupt or a placeholder.")
    else:
        print("      --- Error: File not found after download.")

    # Download SpaCy Model 
    print("--- Checking SpaCy model...")
    !{sys.executable} -m spacy download en_core_web_lg

    # Verify Import 
    print("\n--- Verifying imports...")
    repo_root = os.path.abspath("/sam3-toolkit")

    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
        print(f"--- Added '{repo_root}' to sys.path")

    practical_dir = os.path.join(repo_root, "practical")
    if practical_dir not in sys.path:
        sys.path.insert(0, practical_dir)
        print(f"--- Added '{practical_dir}' to sys.path")

    # Verify all dependencies
    print("\n--- Retrying imports...")
    try:
        import sam3
        print("--- 'sam3' library loaded!")
    except ImportError as e:
        print(f"      --- Failed to load sam3: {e}")
        print("       --- Attempting hard install fix...")
        !pip install "{repo_root}"
        import sam3

    try:
        import models
        print("--- 'models.py' loaded!")
    except ImportError as e:
        print(f"      --- Failed to load models: {e}")

    !pip uninstall numpy -y
    !{sys.executable} -m pip install "numpy<2.0"

    # Check GPU
    import torch
    print(f"--- GPU Available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"--- GPU Name: {torch.cuda.get_device_name(0)}")

    if os.path.exists(practical_dir):
        os.chdir(practical_dir)
        print(f"--- Current Directory: {os.getcwd()}")


---

<font color="#117A65" size='6px'>Loading Core Modules & Dependencies</font>

<font color="#D68910" size='4px'>*semantic_edge:*</font> Contains the main pipeline for generate edge and boundary of objects.

<font color="#D68910" size='4px'>*visualization:*</font> Handles the drawing of bounding boxes, masks, and legends.



In [ ]:
import warnings
import torch
import os
import sys

root = os.path.abspath(os.path.join(os.getcwd(), ".."))

sys.path.append(root)

if os.path.basename(os.getcwd()) == "ipynb":
    os.chdir(root)
    
import semantic_edge
import visualization

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

In [ ]:
if COLAB:
    IMAGE_FILENAME = "images/sa_717.jpg"
    IMAGE_PATH = os.path.join(repo_root, "images", IMAGE_FILENAME)

    if not os.path.exists(IMAGE_PATH):
        alt_path = os.path.join(repo_root, IMAGE_FILENAME)
        if os.path.exists(alt_path):
            VIDEO_PATH = alt_path
        else:
            print(f"      --- Warning: Could not find {IMAGE_FILENAME} in {os.path.dirname(IMAGE_PATH)}")
else:        
    IMAGE_PATH = "images/sa_717.jpg"

```Python
# From configs.py 
LLM_CAPTION_MODEL_NAME = "gpt-5-chat"
SAM3_CONF_IMAGE_FOR_COUNTING = 0.2
```

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

class MockArgs:
    def __init__(self, mode, output_dir, api_key=None, base_url=None):
        self.mode = mode
        self.output = output_dir
        self.api_key = api_key
        self.base_url = base_url


output_directory = "notebook_outputs"
os.makedirs(output_directory, exist_ok=True)

args = MockArgs(mode="blip", output_dir=output_directory)

# for LLM mode:
# args = MockArgs(
#     mode="llm",
#     output_dir=output_directory,
#     api_key="api_key",
#     base_url="base_url"
# )

---

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(20,10))
plt.imshow(plt.imread(IMAGE_PATH))
plt.axis('off')

<font color="#D35400" size='6px'>Execution1: BLIP Automatic Detection</font>

This cell runs the Standard Automatic Mode. By using <font color="#D35400" size='4px'>args.mode='blip'</font>, employ a locally hosted Vision-Language Model to "look" at image and figure out what's inside without needing external APIs.

The function <font color="#D35400"><i>get_classes_blip</i></font> orchestrates a smart scanning process:

<font color="#D35400" size='4px'><i>Smart Cropping:</i></font> To ensure don't miss small details, the image is sliced into multiple strategic crops (zoom-ins) using <font color="#D35400"><i>visualization.get_image_crops</i></font>. This allows the AI to focus on specific regions rather than just the cluttered whole.

<font color="#D35400" size='4px'><i>Local Captioning:</i></font> Runs the <font color="#c5885f"><i>BlipForConditionalGeneration</i></font> model on every single crop. It generates a descriptive sentence for each section,  creating a textual map of the image content.

<font color="#D35400" size='4px'><i>Noun Extraction:</i></font> All these captions are stitched together. then uses a standard NLP library (SpaCy) to parse this text and extract the specific nouns (e.g., "car", "dog"), creating a clean target list for SAM3 to segment.

<font color="#D35400" size='4px'><i>Edge Generation:</i></font> Finally, to create the edge map, the binary masks are processed using <font color="#c5885f"><i>cv2.morphologyEx</i></font>(Morphological Gradient) to isolate and draw the precise boundary pixels of every detected object.

In [ ]:
output_informations = semantic_edge.generate_semantic_edges(IMAGE_PATH, args)

```python
# ...Called when visualizing 'Semantic Edges' mode...
# 1. Initialize Empty Canvas
h, w = image_rgb.shape[:2]
edge_bw = np.zeros((h, w), dtype=np.uint8)
edge_color = np.zeros((h, w, 3), dtype=np.uint8)

# 2. Define Gradient Kernel (Controls edge thickness)
thickness = 2
kernel = np.ones((thickness, thickness), np.uint8)

for mask_data in masks:
    # 3. Get Binary Mask (0 or 255)
    mask = mask_data['segmentation'].astype(np.uint8) * 255
    
    # 4. Calculate Morphological Gradient
    # Gradient = Dilation (Expand) - Erosion (Shrink)
    # This leaves only the boundary pixels
    edges = cv2.morphologyEx(mask, cv2.MORPH_GRADIENT, kernel)
    
    # 5. Add to Accumulator (B&W Map)
    edge_bw = cv2.add(edge_bw, edges)
    
    # 6. Colorize Edges (Color Map)
    class_id = mask_data.get('class_id', 0)
    color = class_colors[class_id]  # Get RGB color
    
    # Only paint pixels where edges exist (>0)
    edge_color[edges > 0] = color
```    

<font color="#9584" size='6px'>Visualizing the Results</font>


In [ ]:
visualization.semantic_edges_visualizations(output_informations)

---

<font color="#3c983c" size='6px'>Execution2: LLM-Enhanced Reasoning</font>

This cell activates the LLM Mode. By simply setting <font color="#3c983c" size='4px'>args.mode='llm'</font>, bypass the standard local models and instead use a Multimodal LLM to get a much deeper understanding of the scene.

This function runs on a fast Async Pipeline:

<font color="#CB4335" size='4px'><i>Asyncio Parallelism:</i></font> unlike BLIP (which works sequentially), the function <font color="#CB4335"><i>get_classes_llm</i></font> chops the image into multiple detailed crops using <font color="#CB4335" size='4px'><i>visualization.get_image_crops*</i></font>. It then sends API requests for all those crops at the exact same time, analyzing the whole image in parallel rather than waiting.

<font color="#CB4335" size='4px'><i>Visual Reasoning:</i></font> Instead of a generic "caption", instruct the model to focus strictly on describing physical objects. This allows the system to spot subtle or complex items that standard BLIP models usually miss.

<font color="#CB4335" size='4px'><i>Intelligent Parsing:</i></font> The raw descriptions from the vision model are collected and sent to a second "Parser" LLM. This step, managed by <font color="#CB4335"><i>utils.search_engine_clean_nouns</i></font>, cleans up the text and extracts a neat list of target nouns that act as precise prompts for SAM3.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"


class MockArgs:
    def __init__(self, mode, output_dir, api_key=None, base_url=None):
        self.mode = mode
        self.output = output_dir
        self.api_key = api_key
        self.base_url = base_url


output_directory = "notebook_outputs"
os.makedirs(output_directory, exist_ok=True)

# for LLM mode:
args = MockArgs(
    mode="llm",
    output_dir=output_directory,
    api_key="api_key",
    base_url="base_url"
)

In [ ]:
output_informations = semantic_edge.generate_semantic_edges(IMAGE_PATH, args)

<font color="#9584" size='6px'>Visualizing the Results</font>


In [ ]:
visualization.semantic_edges_visualizations(output_informations)